In [1]:
import sys
import numpy as np
import pandas as pd
from collections import defaultdict
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import RegressionResultsWrapper 

sys.path.append('..')
from modules.source import Sources, parquet_monthly

CHANNELS: dict[str, tuple[str]] = {
  "Rate Differential": (Sources.rate_differential,),
  "Global Risk Appetite": (Sources.VIX,),
  "Global Credit Condition": (Sources.DXY, Sources.UST_10Y),
  "Commodity": (Sources.crude_palm_oil, Sources.brent_oil),
  "Domestic Baseline": (Sources.cpi_inflation, Sources.IPI)
}
ALL_PREDICTORS = [v for vlist in CHANNELS.values() for v in vlist]
DEPENDENT_VARS = [Sources.KLCI, Sources.financials, Sources.energy, Sources.industrial_products, Sources.plantation, Sources.REITs, Sources.technology]

df_monthly = pd.read_parquet(parquet_monthly).xs("change", axis=1, level=1)

def fit_hac_model(target, rhs_vars, data) -> RegressionResultsWrapper:
  formula = f"{target} ~ {' + '.join(rhs_vars)}"
  L = int(np.floor(4 * (len(data) / 100) ** (2 / 9)))
  return smf.ols(formula, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": L})

full_models, fit_dict, partialr2_dict = {}, {}, {}
wald_dict = defaultdict(dict)

for d_var in DEPENDENT_VARS:
  data = df_monthly[[d_var] + ALL_PREDICTORS].dropna()
  m_full = fit_hac_model(d_var, ALL_PREDICTORS, data)
  full_models[d_var] = m_full

  fit_dict[d_var] = {
    ("Full R2"): round(m_full.rsquared, 4),
    ("Adj. R2"): round(m_full.rsquared_adj, 4),
  }

  partialr2_dict[d_var] = {}
  for ch_name, ch_vars in CHANNELS.items():
    for var in ch_vars:
      rhs_drop = [x for x in ALL_PREDICTORS if x != var] # exclude the current predictor 
      m_drop = fit_hac_model(d_var, rhs_drop, data)
      partialr2_dict[d_var][(ch_name, var)] = round(m_full.rsquared - m_drop.rsquared, 4)

  for ch_name, ch_vars in CHANNELS.items():
    hypothesis = ", ".join(f"{v} = 0" for v in ch_vars)
    w_res = m_full.wald_test(hypothesis, use_f=True, scalar=True)

    wald_dict[ch_name][(d_var, "F-Stat")] = round(float(w_res.statistic), 4)
    wald_dict[ch_name][(d_var, "p-value")] = round(float(w_res.pvalue), 4)

df_fit_matrix = pd.DataFrame(fit_dict)
df_wald_matrix = pd.DataFrame(wald_dict).T
df_partialr2 = pd.DataFrame(partialr2_dict)

### 1. ADF (Stationary), VIF (Multicollinearity)

In [10]:
print("Diagnostic Assumptions (ADF, VIF):")

diagnostic_dict = {}
for col in ALL_PREDICTORS + DEPENDENT_VARS:
  stat, p = adfuller(df_monthly[col].dropna(), autolag="AIC", result_object=True)[:2]
  diagnostic_dict[col] = {
    ("ADF", "score"): round(float(stat), 4),
    ("ADF", "p-val"): round(float(p), 4)
  }

exog_matrix = full_models[DEPENDENT_VARS[0]].model.exog
exog_names = full_models[DEPENDENT_VARS[0]].model.exog_names
for i, var_name in enumerate(exog_names):
  if var_name != "Intercept":
    vif_score = variance_inflation_factor(exog_matrix, i)
    diagnostic_dict[var_name][("VIF", "score")] = round(float(vif_score), 4)

diagnostic_dict = pd.DataFrame(diagnostic_dict).T
diagnostic_dict.columns.names = ["Test", ""]
print(diagnostic_dict)

Diagnostic Assumptions (ADF, VIF):
Test                     ADF             VIF
                       score   p-val   score
FFR_OPR_diff         -4.7400  0.0001  1.1604
VIX                  -5.6326  0.0000  1.1801
DXY                 -11.4835  0.0000  1.3606
UST_10Y             -10.8911  0.0000  1.4257
CPO                  -9.1417  0.0000  1.0780
Oil                  -8.5870  0.0000  1.4882
cpi_inflation        -9.3565  0.0000  1.2212
IPI                  -4.5511  0.0002  1.2895
KLCI                -13.1469  0.0000     NaN
financials          -10.4197  0.0000     NaN
energy              -11.0151  0.0000     NaN
industrial_products -10.0120  0.0000     NaN
plantation          -13.8698  0.0000     NaN
REITs               -14.8697  0.0000     NaN
technology           -4.6504  0.0001     NaN


### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [5]:
print("Model Fit:")
print(df_fit_matrix.to_string())
print("\nChannel Variance Decomposition (PARTIAL R²):")
print(df_partialr2.groupby(level=0, sort=False).sum().to_string())

Model Fit:
           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Full R2  0.3456      0.2849  0.2416               0.1462      0.1297  0.1801      0.1666
Adj. R2  0.3009      0.2360  0.1898               0.0879      0.0572  0.1240      0.1096

Channel Variance Decomposition (PARTIAL R²):
                           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Rate Differential        0.0020      0.0000  0.0043               0.0074      0.0330  0.0248      0.0026
Global Risk Appetite     0.0404      0.0321  0.0008               0.0035      0.0001  0.0323      0.1159
Global Credit Condition  0.1354      0.1358  0.0314               0.0411      0.0215  0.0551      0.0011
Commodity                0.0284      0.0181  0.0934               0.0369      0.0555  0.0429      0.0086
Domestic Baseline        0.0046      0.0013  0.0199               0.0040      0.0082  0.0063      0.0017


### 5. INFERENCE & FEATURE IMPORTANCE

In [8]:
print("Channel Wald Joint Significance Tests:")
print(df_wald_matrix.to_string())

Channel Wald Joint Significance Tests:
                            KLCI         financials          energy         industrial_products         plantation           REITs         technology        
                          F-Stat p-value     F-Stat p-value  F-Stat p-value              F-Stat p-value     F-Stat p-value  F-Stat p-value     F-Stat p-value
Rate Differential         0.2682  0.6055     0.0005  0.9831  0.7446  0.3899              0.6550  0.4200     3.9049  0.0510  2.1421  0.1460     0.6442  0.4238
Global Risk Appetite     14.4102  0.0002     6.4456  0.0124  0.0489  0.8254              0.5523  0.4589     0.0236  0.8781  4.5991  0.0341    10.8959  0.0013
Global Credit Condition  14.1406  0.0000     6.3688  0.0024  2.2882  0.1060              3.5066  0.0332     1.3929  0.2533  3.4298  0.0357     0.0895  0.9145
Commodity                 2.7319  0.0692     1.7771  0.1737  1.7925  0.1711              3.6971  0.0277     6.7333  0.0018  2.9270  0.0575     1.2543  0.2891
Domestic Base